In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/habibaahmed13/cvfile/cv.pdf


In [6]:
!pip install -U langchain langchain-community langchain-core transformers==4.52.4


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 92.8 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 4.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.8/572.8 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.0/256.0 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 85.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.4/162.4 kB 10.3 MB/s eta 0:00:00
  Attempting uninstall: request

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate
import torch
import re

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

def generate_text(prompt, max_length=1000, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00005-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [8]:
name_schema = ResponseSchema(name="full_name", description="The candidate's full name (string)")
email_schema = ResponseSchema(name="email", description="The candidate's email address (string)")
education_schema = ResponseSchema(name="education", description="A list of dictionaries with keys: degree, institution, year")
skills_schema = ResponseSchema(name="skills", description="A list of skills as strings")
experience_schema = ResponseSchema(name="experience", description="A list of dictionaries with keys: role, company, years")

response_schemas = [
    name_schema, 
    email_schema, 
    education_schema, 
    skills_schema, 
    experience_schema
]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

In [9]:
hr_parsing_template = """
You are an HR assistant that parses a short resume snippet into a clean JSON object.

Extract the full name, email, education history, skills, and experience.

Respond ONLY in JSON format as follows:
{format_instructions}

Sample Input Text:
"{user_input}"
"""

# The CV text from the lab PDF
cv_text = """John Smith - john.smith@email.com
Education: B.Sc. Computer Science, MIT, 2020
Skills: Python, Machine Learning, Data Analysis
Experience:
- Software Engineer at Google (2020-2023)
- Data Scientist at OpenAl (2023-Present)"""

prompt = PromptTemplate(
    template=hr_parsing_template,
    input_variables=["user_input", "format_instructions"]
).format(user_input=cv_text, format_instructions=format_instructions)

# Generate the response
response = generate_text(prompt, max_length=1000)[0]
print("Raw Model Output:\n", response)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Raw Model Output:
 
You are an HR assistant that parses a short resume snippet into a clean JSON object.

Extract the full name, email, education history, skills, and experience.

Respond ONLY in JSON format as follows:
The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"full_name": string  // The candidate's full name (string)
	"email": string  // The candidate's email address (string)
	"education": string  // A list of dictionaries with keys: degree, institution, year
	"skills": string  // A list of skills as strings
	"experience": string  // A list of dictionaries with keys: role, company, years
}
```

Sample Input Text:
"John Smith - john.smith@email.com
Education: B.Sc. Computer Science, MIT, 2020
Skills: Python, Machine Learning, Data Analysis
Experience:
- Software Engineer at Google (2020-2023)
- Data Scientist at OpenAl (2023-Present)"
```json
{
	"full_name": "John Smith",
	"email"

In [10]:
import re

def extract_json_block(text):
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)

    return f"```json\n{matches[-1]}\n```"

In [11]:
# Extract JSON block and parse it into a dictionary
json_text = extract_json_block(response)
final_output = output_parser.parse(json_text)

print("\nParsed Python Dictionary:")
print(final_output)


Parsed Python Dictionary:
{'full_name': 'John Smith', 'email': 'john.smith@email.com', 'education': [{'degree': 'B.Sc. Computer Science', 'institution': 'MIT', 'year': 2020}], 'skills': ['Python', 'Machine Learning', 'Data Analysis'], 'experience': [{'role': 'Software Engineer', 'company': 'Google', 'years': '2020-2023'}, {'role': 'Data Scientist', 'company': 'OpenAl', 'years': '2023-Present'}]}


In [12]:
!pip install pypdf

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [13]:
# /kaggle/input/datasets/habibaahmed13/cvfile/cv.pdf
from pypdf import PdfReader

pdf_path = "/kaggle/input/datasets/habibaahmed13/cvfile/cv.pdf" 

reader = PdfReader(pdf_path)
cv_text_extracted = ""

for page in reader.pages:
    cv_text_extracted += page.extract_text() + "\n"

print("Extracted Text Preview:")
print(cv_text_extracted[:500]) 

Extracted Text Preview:
Habiba Ahmed Hashim 
 
 
 
 
Cairo, Egypt • habibaahmed1321@gmail.com • 01094883794 • linkedin.com/in/habibaahmed1321 
Education 
Ain Shams University  
Bachelor’s Degree, Computer and Systems Engineering.                                                            Expected Graduation: 2027  
GPA: 2.97 / 4.0 | Grade: B- 
Experience 
 National Telecommunication Institute (NTI)                                                                                            Cairo, Egypt 
Full-stack Develo


In [14]:
prompt = PromptTemplate(
    template=hr_parsing_template,
    input_variables=["user_input", "format_instructions"]
).format(user_input=cv_text_extracted, format_instructions=format_instructions)

response = generate_text(prompt, max_length=1500)[0]
print("Raw Model Output:\n", response)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Raw Model Output:
 
You are an HR assistant that parses a short resume snippet into a clean JSON object.

Extract the full name, email, education history, skills, and experience.

Respond ONLY in JSON format as follows:
The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"full_name": string  // The candidate's full name (string)
	"email": string  // The candidate's email address (string)
	"education": string  // A list of dictionaries with keys: degree, institution, year
	"skills": string  // A list of skills as strings
	"experience": string  // A list of dictionaries with keys: role, company, years
}
```

Sample Input Text:
"Habiba Ahmed Hashim 
 
 
 
 
Cairo, Egypt • habibaahmed1321@gmail.com • 01094883794 • linkedin.com/in/habibaahmed1321 
Education 
Ain Shams University  
Bachelor’s Degree, Computer and Systems Engineering.                                                            Expec

In [15]:
json_text = extract_json_block(response)
final_parsed_cv = output_parser.parse(json_text)

print("\nMy CV Parsed Output:")
print(final_parsed_cv)


My CV Parsed Output:
{'full_name': 'Habiba Ahmed Hashim', 'email': 'habibaahmed1321@gmail.com', 'education': [{'degree': 'Bachelor’s Degree, Computer and Systems Engineering', 'institution': 'Ain Shams University', 'year': 'Expected Graduation: 2027'}], 'skills': ['C++', 'Python', 'Java', 'SQL', 'HTML', 'CSS', 'JavaScript', 'PHP', 'MySQL', 'Laravel', 'Scikit-learn', 'Data Preprocessing', 'Model Evaluation', 'OOP', 'Data Structures & Algorithms', 'Operating Systems', 'Git', 'GitHub', 'VS Code', 'SQL Server', 'Arabic (Native)', 'English (Proficient)'], 'experience': [{'role': 'Full-stack Development trainee', 'company': 'National Telecommunication Institute (NTI)', 'years': 'Jun 2025 – Jul 2025'}]}


In [16]:
!pip install -q fastapi uvicorn pyngrok python-multipart

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [18]:
import json
import re
import threading
 
import torch
import uvicorn
from fastapi import FastAPI
from kaggle_secrets import UserSecretsClient
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel
from pyngrok import ngrok
 
# Token comes from Kaggle -> Add-ons -> Secrets (name: NGROK_TOKEN)
ngrok.set_auth_token(UserSecretsClient().get_secret("NGROK_TOKEN"))
 
app = FastAPI(title="CV Parser API")
gpu_lock = threading.Lock()  # one generation at a time
 
 
class CVRequest(BaseModel):
    text: str
 
 
def extract_json(text: str):
    """Pull the JSON object out of the model output. Returns dict or None."""
    blocks = re.findall(r"```json\s*(.*?)\s*```", text, re.DOTALL)
    if blocks:
        candidate = blocks[-1]
    else:
        start, end = text.find("{"), text.rfind("}")
        if start == -1 or end == -1:
            return None
        candidate = text[start : end + 1]
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        return None
 
 
@app.get("/health")
def health():
    return {"status": "ok"}
 
 
@app.post("/parse")
def parse_cv(req: CVRequest):
    prompt = PromptTemplate(
        template=hr_parsing_template,
        input_variables=["user_input", "format_instructions"],
    ).format(user_input=req.text[:6000], format_instructions=format_instructions)
 
    with gpu_lock:
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=900,
                do_sample=True,
                top_k=50,
                top_p=0.95,
                temperature=0.3,
            )
 
    # decode ONLY the newly generated tokens (not the prompt)
    new_tokens = output[0][inputs["input_ids"].shape[1] :]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return {"parsed": extract_json(raw), "raw": raw}
 
 
server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
 
tunnel = ngrok.connect(8000)
print("=" * 60)
print("PUBLIC URL (paste this into Streamlit):")
print(tunnel.public_url)
print("=" * 60)

ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use


PUBLIC URL (paste this into Streamlit):
https://twerp-delouse-willing.ngrok-free.dev


In [ ]:
import time

while True:
    time.sleep(60)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
